# Exploration of WVV ÖPNV data using the GTFS database

No real goal, just exploring the data while printing, plotting and saving as much relevant information as possible.

In [80]:
import pandas as pd

pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## GTFS Static - Extraction

### Download and preparation

In [119]:
import requests
import zipfile
import io
import os


# The URL for the GTFS feed
url = "https://download.gtfs.de/germany/nv_free/latest.zip"
  
# Download the file
response = requests.get(url)

# Check if the download was successful
if response.status_code == 200:
    print("Download complete. Extracting files...")
    
    # Extract the content
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall("../data_static")
        
    print(f"Success! Files extracted to: {os.path.abspath("../data_static")}")
else:
    print(f"Failed to download. Status code: {response.status_code}")


Download complete. Extracting files...
Success! Files extracted to: c:\Users\Raphael Teller\Documents\Code\Python\mo-buzz\data_static


### Extracting all routes managed by WVV

In [124]:
df_agency = pd.read_csv("../data_static/agency.txt")
df_wvv = df_agency.loc[df_agency["agency_name"] == "Würzburger Verkehrsverbund"].reset_index(drop=True)
df_wvv

,agency_id,agency_name,agency_url,agency_timezone,agency_lang
0,335,Würzburger Verkehrsverbund,https://www.bahn.de,Europe/Berlin,de


In [125]:
df_routes = pd.read_csv("../data_static/routes.txt")

wvv_id = df_wvv.loc[0, "agency_id"]
df_routes_wvv = df_routes.loc[df_routes["agency_id"] == wvv_id]
df_routes_wvv = df_routes_wvv.drop("agency_id", axis=1).reset_index(drop=True)

print(f"Total number of routes managed by WVV: {df_routes_wvv.shape[0]}")

df_routes_wvv.head()


Total number of routes managed by WVV: 91


,route_long_name,route_short_name,route_type,route_id,route_color,route_text_color
0,NaN,1,0,10266,NaN,NaN
1,NaN,10,3,14694,NaN,NaN
2,NaN,11,3,8495,NaN,NaN
3,NaN,114,3,1793,NaN,NaN
4,NaN,12,3,14423,NaN,NaN


In [126]:
print(f"Unique long route names: {df_routes_wvv["route_long_name"].unique()}")
print(f"Unique route colors: {df_routes_wvv["route_color"].unique()}")
print(f"Unique route text colors: {df_routes_wvv["route_text_color"].unique()}")

Unique long route names: [nan]
Unique route colors: <StringArray>
[nan]
Length: 1, dtype: str
Unique route text colors: <StringArray>
[nan]
Length: 1, dtype: str


As shown above `route_long_name`, `route_color` and `route_text_color` contains no information, hence will be dropped.

In [127]:
df_routes_wvv = df_routes_wvv.drop(["route_long_name", "route_color", "route_text_color"], axis=1)

df_routes_wvv.to_csv("../data_extract/routes_wvv.csv", index=False)

df_routes_wvv.head()

,route_short_name,route_type,route_id
0,1,0,10266
1,10,3,14694
2,11,3,8495
3,114,3,1793
4,12,3,14423


### Differentiating route types

In [128]:
print(f"Unique route types: {df_routes_wvv["route_type"].unique()}")

Unique route types: [0 3]


In [129]:
df_routes_wvv_0 = df_routes_wvv.loc[df_routes_wvv["route_type"] == 0]
df_routes_wvv_0

,route_short_name,route_type,route_id
0,1,0,10266
21,3,0,9338
25,4,0,4186
51,5,0,13453
57,53,0,3148
58,54,0,2009


In [131]:
df_routes_wvv_3 = df_routes_wvv.loc[df_routes_wvv["route_type"] == 3]
df_routes_wvv_3.head()

,route_short_name,route_type,route_id
1,10,3,14694
2,11,3,8495
3,114,3,1793
4,12,3,14423
5,13,3,20601


While it is not specified to what the `route_type` columns refers to, I highly assume that `route_type = 0` are Straßenbahnen, while `route_type = 3` are buses.

Wierdly, the Straßenbahnen `53` and `54` are not regular Straßenbahnen and the Straßebahn with number `2` is missing.

### Extracting all the trips managed by WVV

In [132]:
df_trips = pd.read_csv("../data_static/trips.txt")
df_trips_wvv = df_trips.loc[df_trips["route_id"].isin(df_routes_wvv["route_id"])]

print(f"Total number of trips managed by WVV: {df_trips_wvv.shape[0]}")

df_trips_wvv.head()

Total number of trips managed by WVV: 7260


,route_id,service_id,trip_id
16569,10266,838,332090
32194,1050,1205,1112520
32195,1050,1205,1157188
32196,1050,1205,1210708
32197,1050,1205,1230995


In [133]:
df_trips_wvv = df_trips_wvv.merge(df_routes_wvv, on="route_id")

df_trips_wvv.to_csv("../data_extract/trips_wvv.csv", index=False)

df_trips_wvv.head()

,route_id,service_id,trip_id,route_short_name,route_type
0,10266,838,332090,1,0
1,1050,1205,1112520,551,3
2,1050,1205,1157188,551,3
3,1050,1205,1210708,551,3
4,1050,1205,1230995,551,3


### Extracting all the stop times managed by WVV

In [134]:
# Reading this file takes a few seconds, due to its size: hence seperate cell
df_stoptimes = pd.read_csv("../data_static/stop_times.txt")

In [135]:
df_stoptimes_wvv = df_stoptimes.merge(df_trips_wvv, on="trip_id", how="inner")

df_stoptimes_wvv.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type,route_id,service_id,route_short_name,route_type
0,332090,12:36:00,12:36:00,679255,0,NaN,NaN,10266,838,1,0
1,332090,12:36:42,12:37:00,648315,1,NaN,NaN,10266,838,1,0
2,332090,12:37:42,12:38:00,291669,2,NaN,NaN,10266,838,1,0
3,332090,12:38:42,12:39:00,491827,3,NaN,NaN,10266,838,1,0
4,332090,12:40:42,12:41:00,504277,4,NaN,NaN,10266,838,1,0


In [136]:
print(f"Unique pickup types: {df_stoptimes_wvv["pickup_type"].unique()}")
print(f"Unique dropoff types: {df_stoptimes_wvv["drop_off_type"].unique()}")
print(f"Unique service IDs: {df_stoptimes_wvv["service_id"].unique()}")

Unique pickup types: [nan  1.]
Unique dropoff types: [nan]
Unique service IDs: [ 838 1205 1500  137   70  693 1590  725  247 1067 1479  655  998 1087
  312 1417   78  162 1772 1056  974  962 1262  566 1390 1524   35  339
 1808  166 1413 1030 1121 1443 1605  401  250 1433  221  769 1464 1637
  372  176 1473]


As shown above `drop_off_type` contains no information, hence will be dropped.

In [137]:
df_stoptimes_wvv = df_stoptimes_wvv.drop("drop_off_type", axis=1)

df_stoptimes_wvv.to_csv("../data_extract/stoptimes_wvv.csv", index=False)

df_stoptimes_wvv.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,route_id,service_id,route_short_name,route_type
0,332090,12:36:00,12:36:00,679255,0,NaN,10266,838,1,0
1,332090,12:36:42,12:37:00,648315,1,NaN,10266,838,1,0
2,332090,12:37:42,12:38:00,291669,2,NaN,10266,838,1,0
3,332090,12:38:42,12:39:00,491827,3,NaN,10266,838,1,0
4,332090,12:40:42,12:41:00,504277,4,NaN,10266,838,1,0


### Extracting all the stops managed by WVV

In [138]:
df_stops = pd.read_csv("../data_static/stops.txt")

stops_wvv_ids = df_stoptimes_wvv["stop_id"].unique()
df_stops_wvv = df_stops.loc[df_stops["stop_id"].isin(stops_wvv_ids)].reset_index(drop=True)

print(f"Number of stops managed by WVV: {df_stops_wvv.shape[0]}")

df_stops_wvv.head()

Number of stops managed by WVV: 1622


,stop_name,parent_station,stop_id,stop_lat,stop_lon,location_type,platform_code
0,Acholshausen Acholshausen,259853.0,244818,49.645176,9.997971,NaN,NaN
1,Acholshausen Acholshausen,259853.0,597760,49.645252,9.998132,NaN,NaN
2,Albertsh. Fuchsstadter Str.,662844.0,146912,49.693928,9.932169,NaN,12
3,Albertsh. Lindflurer Str.,412919.0,552381,49.694230,9.927417,NaN,12
4,Albertsh. Lindflurer Str.,412919.0,569884,49.694195,9.927336,NaN,11


In [139]:
print(f"Unique location types: {df_stops_wvv["location_type"].unique()}")

Unique location types: [nan]


As shown above `location_type` contains no information, hence will be dropped.

In [140]:
df_stops_wvv = df_stops_wvv.drop("location_type", axis=1)

df_stops_wvv.to_csv("../data_extract/stops_wvv.csv", index=False)

df_stops_wvv.head()

,stop_name,parent_station,stop_id,stop_lat,stop_lon,platform_code
0,Acholshausen Acholshausen,259853.0,244818,49.645176,9.997971,NaN
1,Acholshausen Acholshausen,259853.0,597760,49.645252,9.998132,NaN
2,Albertsh. Fuchsstadter Str.,662844.0,146912,49.693928,9.932169,12
3,Albertsh. Lindflurer Str.,412919.0,552381,49.694230,9.927417,12
4,Albertsh. Lindflurer Str.,412919.0,569884,49.694195,9.927336,11


## GTFS Static - Plotting

In [1]:
from folium_plots import plot_stops_map

### Vizualizing all Stops

In [4]:
df_stops_wvv = pd.read_csv("../data_extract/stops_wvv.csv")

plot_stops_map(df_stops_wvv, f"stops_map")

Map saved as stops_map.html. Open this file in your web browser!


Since there are so many stops, this is quite laggy. Here we will group all stops that have the same name (eg. Hauptbahnhof ZOB has all platforms as different stops)

In [5]:
df_stops_wvv["mean_lat"] = df_stops_wvv.groupby("stop_name")["stop_lat"].transform("mean")
df_stops_wvv["mean_lon"] = df_stops_wvv.groupby("stop_name")["stop_lon"].transform("mean")

df_stops_wvv_grouped = df_stops_wvv.drop_duplicates(subset="stop_name").reset_index(drop=True)
df_stops_wvv_grouped = df_stops_wvv_grouped[["stop_name", "stop_id", "mean_lat", "mean_lon"]].rename({"mean_lat": "stop_lat", "mean_lon": "stop_lon"}, axis=1)

plot_stops_map(df_stops_wvv_grouped, f"stops_map_grouped")

Map saved as stops_map_grouped.html. Open this file in your web browser!


### Vizualizing how often Stops are visited

In [6]:
df_stoptimes_wvv = pd.read_csv("../data_extract/stoptimes_wvv.csv", dtype={"route_short_name": str})

df_stopvisits = df_stoptimes_wvv.groupby("stop_id").size().reset_index(name="count")
df_stopvisits.head()

df_stopvisits = df_stopvisits.merge(df_stops_wvv_grouped, on="stop_id", how="inner")
df_stopvisits = df_stopvisits[["stop_name", "stop_id", "count", "stop_lat", "stop_lon"]]

min_val = df_stopvisits["count"].min()
max_val = df_stopvisits["count"].max()
        
y_vals = 255 - (df_stopvisits["count"] - min_val) / (max_val - min_val) * 255
df_stopvisits["color"] = y_vals.apply(lambda y: f"rgb(255, {int(y)}, 0)")

plot_stops_map(df_stopvisits, f"stops_map_grouped_heat", heat=True)

Map saved as stops_map_grouped_heat.html. Open this file in your web browser!


# GTFS-RT

### Field Analysis

GTFS RT data is built as a sort of stream, with following hierachy (only a selection):

| Field                                                         | Description                                                 |
| ----------------------------------------------------          | ----------------------------------------------------------- |
| `entity.trip`                                                 | Information about the trip                                  |
| `entity.trip.trip_id`                                         | Trip ID as in GTFS static (str)                             |
| `entity.trip.route_id`                                        | Route ID as in GTFS static (str)                            |
| `entity.trip.start_time`                                      | The scheduled start time of this trip (HH:MM:SS)            |
| `entity.trip.start_date`                                      | The scheduled start day of this trip (YYYYMMDD)             |
|                                                               |                                                             |
| `entity.vehicle`                                              | Vehicle specific information (e.g., number plate)           |
|                                                               |                                                             |
| `entity.stop_time_update`                                     | Stop time update information                                |
| `entity.stop_time_update.stop_sequence`                       | Stop sequence as in GTFS static (uint32)                    |
| `entity.stop_time_update.stop_id`                             | Stop ID as in GTFS static (str)                             |
| `entity.stop_time_update.departure_occupancy_status`          | Vehicle occupancy status (enum)                             |
| `entity.stop_time_update.schedule_relationship`               | Relationship to static schedule (e.g., skipped stop) (enum) |
|                                                               |                                                             |
| `entity.stop_time_update.arrival`                             | Arrival time information                                    |
| `entity.stop_time_update.departure`                           | Departure time information                                  |
| `entity.stop_time_update.{arrival,departure}.delay`           | Delay in seconds (int32)                                    |
| `entity.stop_time_update.{arrival,departure}.time`            | Actual time in POSIX (int64)                                |
| `entity.stop_time_update.{arrival,departure}.scheduled_time`  | Scheduled time in POSIX (int64)                             |
| `entity.stop_time_update.{arrival,departure}.uncertainty`     | Uncertainty (0 = certain) (int32)                           |
|                                                               |                                                             |
| `entity.timestamp`                                            | Timestamp of measurement in POSIX time (uint64)             |
| `entity.delay`                                                | Experimental delay in seconds (int32)                       |
| `entity.trip_properties`                                      | Experimental message, not that relevant                     |

Very important to note, not all fields must necessarily be present.

### Code

In [141]:
import requests
from datetime import datetime as dt
from google.transit import gtfs_realtime_pb2

URL = "https://realtime.gtfs.de/realtime-free.pb"

In [142]:
def get_gtfs_rt_delays(url):
    # Fetch the realtime feed
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed to fetch data: {response.status_code}")
        return pd.DataFrame()

    # Parse the protobuf data
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    # Extract relevant data into a list of dictionaries
    data = []
    for entity in feed.entity:
        if entity.HasField("trip_update"):
            message = entity.trip_update

            trip_id = message.trip.trip_id
            num_stop_time_updates = len(message.stop_time_update)

            for stu in message.stop_time_update:
                row = {}

                row["trip_id"]                  = trip_id
                row["num_stop_time_updates"]    = num_stop_time_updates
                row["stop_sequence"]            = stu.stop_sequence
                row["stop_id"]                  = stu.stop_id
                row["arrival_delay"]            = stu.arrival.delay
                row["departure_delay"]          = stu.departure.delay
                row["arrival_time"]             = stu.arrival.time
                row["departure_time"]           = stu.departure.time

                data.append(row)

    # Convert to pandas DataFrame
    return pd.DataFrame(data)

In [143]:
now = dt.now().strftime("%Y%m%d_%H%M%S")
df_rt = get_gtfs_rt_delays(URL)
df_rt.to_csv(f"../data_rt/rt_snapshot_{now}.csv", index=False)

In [152]:
SNAPSHOT_TIME = "20260424_171807"

df_rt = pd.read_csv(f"../data_rt/rt_snapshot_{SNAPSHOT_TIME}.csv")
#df_stoptimes_wvv = pd.read_csv("../data_extract/stoptimes_wvv.csv", dtype={"route_short_name": str})
df_stops_wvv = pd.read_csv("../data_extract/stops_wvv.csv")[["stop_id", "stop_name"]]

#df_stoptimes_wvv = df_stoptimes_wvv[["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence", "route_short_name", "route_type"]]
#df_stoptimes_wvv = df_stoptimes_wvv.rename({"arrival_time": "at_schedule", "departure_time": "dt_schedule"}, axis=1)

df_rt = df_rt.rename({"arrival_time": "at_rt", "departure_time": "dt_rt"}, axis=1)
df_rt["at_rt"] = pd.to_datetime(df_rt["at_rt"], unit="s")
df_rt["dt_rt"] = pd.to_datetime(df_rt["dt_rt"], unit="s")

df_rt_wvv = df_rt.merge(df_stops_wvv, on=["stop_id"], how="inner")

df_rt_wvv.head()


,trip_id,num_stop_time_updates,stop_sequence,stop_id,arrival_delay,departure_delay,at_rt,dt_rt,stop_name
0,973362,33,0,76686,0,154,1970-01-01 00:00:00,2026-04-24 13:54:34,Würzburg Bismarckstraße
1,973362,33,1,98844,667,675,2026-04-24 14:06:07,2026-04-24 14:06:15,Würzburg Weißenburgstraße
2,973362,33,2,291239,703,710,2026-04-24 14:08:43,2026-04-24 14:08:50,Würzburg Vogel-Verlag
3,973362,33,3,16646,810,829,2026-04-24 14:13:30,2026-04-24 14:13:49,Zell/Wasserwerk
4,973362,33,4,270990,831,831,2026-04-24 14:17:51,2026-04-24 14:17:51,Margetshöchheim Birkachstraße


### Single route delay exploration at snapshot time

In [ ]:
TRIP_ID = 973362

df_rt_wvv_route = df_rt_wvv.loc[df_rt_wvv["trip_id"] == TRIP_ID]

df_rt_wvv_route.head()

NameError: name 'trip_id' is not defined